In [3]:
# ============================================================
# Parameter scan:
#   sigma_base = 0.5, 0.6, ..., 1.5       (11 values)
#   L_var      = 0, 0.1, 0.2              (3 values)
#   max_inter  = 1, 5, 10                 (3 values)
#
# Each parameter combination:
#   10 random seeds
#   Treatment + dilution control
#
# Total:
#   11 × 3 × 3 = 99 scenarios
#   99 × 10 = 990 scenario-seed combinations
#   990 × 2 conditions = 1,980 time courses
# ============================================================

import numpy as np
from scipy.integrate import odeint


# ============================================================
# 1. Utility functions
# ============================================================

def generate_valid_random(N):
    """
    Generate N values from a standard normal distribution.

    The complete random vector is accepted only when:
        abs(mean) <= 0.1
        standard deviation <= 1.2

    This follows the random-number filtering used in the
    previous no-debris model.
    """
    while True:
        r = np.random.randn(N)

        if (
            abs(np.mean(r)) <= 0.1
            and np.std(r) <= 1.2
        ):
            return r


def param_generator_lv(
    N,
    connectedness,
    neg_frac,
    max_neg,
    max_pos
):
    """
    Generate a Lotka-Volterra interaction matrix.

    Parameters
    ----------
    N : int
        Number of species.

    connectedness : float
        Fraction of nonzero interspecies interactions.
        connectedness = 1 means fully connected.

    neg_frac : float
        Fraction of nonzero interactions that are negative.

    max_neg : float
        Maximum absolute strength of negative interactions.
        Negative interactions are sampled from:
            Uniform(-max_neg, 0)

    max_pos : float
        Maximum strength of positive interactions.
        Positive interactions are sampled from:
            Uniform(0, max_pos)

    Returns
    -------
    list
        [
            complete interaction matrix,
            negative interaction matrix,
            positive interaction matrix
        ]
    """

    total_off_diagonal = N * (N - 1)

    # Random ordering of all off-diagonal interaction positions
    inds_perm = np.random.permutation(total_off_diagonal)

    # Temporary matrix excluding the diagonal
    inter_init = np.zeros((N, N - 1))

    neg_count = int(
        np.ceil(
            total_off_diagonal
            * connectedness
            * neg_frac
        )
    )

    pos_count = int(
        np.floor(
            total_off_diagonal
            * connectedness
            * (1 - neg_frac)
        )
    )

    # Negative interactions
    if neg_count > 0 and max_neg > 0:
        inter_init.flat[
            inds_perm[:neg_count]
        ] = (
            -np.random.rand(neg_count)
            * max_neg
        )

    # Positive interactions
    if pos_count > 0 and max_pos > 0:
        start = neg_count
        end = neg_count + pos_count

        inter_init.flat[
            inds_perm[start:end]
        ] = (
            np.random.rand(pos_count)
            * max_pos
        )

    # Restore an N × N matrix with zero diagonal
    inter = np.zeros((N, N))

    for i in range(N):
        mask = np.ones(N, dtype=bool)
        mask[i] = False
        inter[i, mask] = inter_init[i, :]

    negative_inter = inter * (inter < 0)
    positive_inter = inter * (inter > 0)

    return [
        inter,
        negative_inter,
        positive_inter
    ]


# ============================================================
# 2. Mechanistic model
# ============================================================

def core_ode_lv_linparam(
        x, t, inter, gamma, beta,
        max_Gs, sigma, lambda_val
):
    # Effective growth rate
    G = (
        max_Gs * (1 - x)
        + inter @ x
    )

    # Only positive growth contributes to
    # growth-dependent lysis
    L = np.maximum(
    sigma * np.maximum(G, 0.0) + lambda_val,
    0.0
    )

    # Net population dynamics
    return x * (G - L)


def calculate_net_growth_rates(
    x,
    params,
    max_Gs,
    sigma,
    lambda_val
):
    """
    Return net population rate:

        N_i = G_i - L_i
    """

    G = (
        max_Gs * (1 - x)
        + params[0] @ x
    )

    L = np.maximum(
    sigma * np.maximum(G, 0.0) + lambda_val,
    0.0
    )

    return G - L


def calculate_effective_growth_rates(
    x,
    params,
    max_Gs
):
    """
    Return effective growth rate:

        G_i = max_G_i × (1 - x_i)
              + sum_j(inter_ij × x_j)
    """

    return (
        max_Gs * (1 - x)
        + params[0] @ x
    )

def calculate_lysis_rates(
        x, params, max_Gs,
        sigma, lambda_val
):
    G = (
        max_Gs * (1 - x)
        + params[0] @ x
    )

    L = np.maximum(
    sigma * np.maximum(G, 0.0) + lambda_val,
    0.0
    )

    return L


# ============================================================
# 3. Treatment simulation
# ============================================================

def simulate_treatment_timecourse(
    x0,
    tend,
    params,
    max_Gs,
    sigma,
    lambda_val,
    treatment_time=4.0
):
    """
    Treatment simulation.

    Phase 1: 0–4 h
        Growth only:
            sigma = 0
            lambda = 0

    Phase 2: 4–11 h
        Growth and lysis:
            sigma = species-specific treatment values
            lambda = species-specific treatment values

    The complete output contains 111 time points:
        0, 0.1, ..., 11 h
    """

    if treatment_time <= 0 or treatment_time >= tend:
        raise ValueError(
            "treatment_time must be between 0 and tend."
        )

    N = len(x0)

    # Complete output time grid
    t = np.linspace(
        0,
        tend,
        int(tend * 10) + 1
    )

    # Include exactly 4 h in phase 1
    t_phase1 = t[t <= treatment_time]

    # Times strictly after 4 h
    t_phase2_output = t[t > treatment_time]

    zero_vector = np.zeros(N)

    # --------------------------------------------------------
    # Phase 1: growth only
    # --------------------------------------------------------

    y1 = odeint(
        core_ode_lv_linparam,
        x0,
        t_phase1,
        args=(
            params[0],
            params[1],
            params[2],
            max_Gs,
            zero_vector,
            zero_vector
        ),
        atol=1e-9,
        rtol=1e-9
    )

    # --------------------------------------------------------
    # Phase 2: treatment begins at 4 h
    # --------------------------------------------------------

    # Add 4 h as the integration starting point
    t_phase2_ode = np.concatenate(
        ([treatment_time], t_phase2_output)
    )

    y2_with_initial = odeint(
        core_ode_lv_linparam,
        y1[-1],
        t_phase2_ode,
        args=(
            params[0],
            params[1],
            params[2],
            max_Gs,
            sigma,
            lambda_val
        ),
        atol=1e-9,
        rtol=1e-9
    )

    # Remove the duplicated 4 h initial row
    y2 = y2_with_initial[1:]

    # Complete population trajectory
    y = np.vstack([y1, y2])

    # Numerical protection against tiny negative abundances
    y = np.maximum(y, 0)

    # --------------------------------------------------------
    # Calculate rates
    # --------------------------------------------------------

    net_growth_rates = np.zeros_like(y)
    effective_growth_rates = np.zeros_like(y)
    lysis_rates = np.zeros_like(y)

    for i, current_time in enumerate(t):

        effective_growth_rates[i] = (
            calculate_effective_growth_rates(
                y[i],
                params,
                max_Gs
            )
        )

        if current_time < treatment_time:
            current_sigma = zero_vector
            current_lambda = zero_vector
        else:
            current_sigma = sigma
            current_lambda = lambda_val

        lysis_rates[i] = calculate_lysis_rates(
            y[i],
            params,
            max_Gs,
            current_sigma,
            current_lambda
        )

        net_growth_rates[i] = (
            effective_growth_rates[i]
            - lysis_rates[i]
        )

    return (
        t,
        y,
        net_growth_rates,
        effective_growth_rates,
        lysis_rates
    )


# ============================================================
# 4. Control simulation
# ============================================================

def simulate_control_timecourse(
    x0,
    tend,
    params,
    max_Gs,
    dilution_time=4.0,
    dilution_factor=25.0
):
    """
    Dilution control.

    Phase 1: 0–4 h
        Growth only.

    At exactly 4 h:
        Every species is diluted by the same factor.

    Phase 2: 4–11 h
        Growth only; no lysis.

    The stored abundance at 4 h is the post-dilution abundance.
    """

    if dilution_factor <= 0:
        raise ValueError(
            "dilution_factor must be greater than 0."
        )

    if dilution_time <= 0 or dilution_time >= tend:
        raise ValueError(
            "dilution_time must be between 0 and tend."
        )

    N = len(x0)

    t = np.linspace(
        0,
        tend,
        int(tend * 10) + 1
    )

    t_phase1 = t[t <= dilution_time]
    t_phase2_output = t[t > dilution_time]

    zero_vector = np.zeros(N)

    # --------------------------------------------------------
    # Phase 1: growth before dilution
    # --------------------------------------------------------

    y1 = odeint(
        core_ode_lv_linparam,
        x0,
        t_phase1,
        args=(
            params[0],
            params[1],
            params[2],
            max_Gs,
            zero_vector,
            zero_vector
        ),
        atol=1e-9,
        rtol=1e-9
    )

    # --------------------------------------------------------
    # Dilution at 4 h
    # --------------------------------------------------------

    y_after_dilution = (
        y1[-1] / dilution_factor
    )

    # Replace the 4 h pre-dilution abundance with the
    # post-dilution abundance
    y1[-1] = y_after_dilution

    # --------------------------------------------------------
    # Phase 2: growth after dilution
    # --------------------------------------------------------

    t_phase2_ode = np.concatenate(
        ([dilution_time], t_phase2_output)
    )

    y2_with_initial = odeint(
        core_ode_lv_linparam,
        y_after_dilution,
        t_phase2_ode,
        args=(
            params[0],
            params[1],
            params[2],
            max_Gs,
            zero_vector,
            zero_vector
        ),
        atol=1e-9,
        rtol=1e-9
    )

    y2 = y2_with_initial[1:]

    y = np.vstack([y1, y2])

    # Numerical protection
    y = np.maximum(y, 0)

    # --------------------------------------------------------
    # Calculate rates: no lysis at any time
    # --------------------------------------------------------

    net_growth_rates = np.zeros_like(y)
    effective_growth_rates = np.zeros_like(y)
    lysis_rates = np.zeros_like(y)

    for i in range(len(t)):

        effective_growth_rates[i] = (
            calculate_effective_growth_rates(
                y[i],
                params,
                max_Gs
            )
        )

        lysis_rates[i] = 0

        net_growth_rates[i] = (
            effective_growth_rates[i]
        )

    return (
        t,
        y,
        net_growth_rates,
        effective_growth_rates,
        lysis_rates
    )


# ============================================================
# 5. Fixed model parameters
# ============================================================

N = 57

# Equal initial abundance across 57 species
# Total initial abundance = 0.01
x0 = np.ones(N) / (100 * N)

tend = 11

# Same intrinsic growth-rate range as the previous
# no-debris simulation
max_Gs = np.linspace(
    0.7,
    1.5,
    N
)

# Baseline lysis rate
lambda_base = 0.1
lambda_val = (
    lambda_base * np.ones(N)
)

# Fully connected interaction network
connectedness = 1.0

# 50% negative and 50% positive interactions
neg_frac = 1

# Treatment/control timing
treatment_time = 4.0
dilution_time = 4.0
dilution_factor = 25.0


# ============================================================
# 6. Parameter scan
# ============================================================

# 0.5, 0.6, ..., 1.4
# np.round avoids floating-point labels such as 0.799999999
sigma_base_values = np.round(
    np.arange(0.5, 1.6, 0.1),
    1
)

L_var_values = [
    0.0,
    0.1,
    0.2
]

max_inter_values = [
    1,
    5,
    10
]

scenario_params = []

for sigma_base in sigma_base_values:
    for L_var in L_var_values:
        for max_inter in max_inter_values:

            scenario_params.append(
                {
                    "sigma_base": float(sigma_base),
                    "L_var": float(L_var),
                    "max_inter": int(max_inter)
                }
            )

n_scenarios = len(scenario_params)
n_seeds = 10

print(
    f"Number of sigma_base values: "
    f"{len(sigma_base_values)}"
)

print(
    f"Number of L_var values: "
    f"{len(L_var_values)}"
)

print(
    f"Number of max_inter values: "
    f"{len(max_inter_values)}"
)

print(
    f"Total scenarios: {n_scenarios}"
)

print(
    f"Scenario-seed combinations: "
    f"{n_scenarios * n_seeds}"
)

print(
    f"Treatment + control timecourses: "
    f"{n_scenarios * n_seeds * 2}"
)


# ============================================================
# 7. Initialize output arrays
# ============================================================

n_time = int(tend * 10) + 1
n_species = N

output_shape = (
    n_time,
    n_species,
    n_seeds,
    n_scenarios
)

# Treatment outputs
outputs = {
    "population_data":
        np.zeros(output_shape),

    "growth_rates":
        np.zeros(output_shape),

    "effective_growth_rates":
        np.zeros(output_shape),

    "lysis_rates":
        np.zeros(output_shape),

    "relative_abundance":
        np.zeros(output_shape)
}

# Control outputs
control_outputs = {
    "population_data":
        np.zeros(output_shape),

    "growth_rates":
        np.zeros(output_shape),

    "effective_growth_rates":
        np.zeros(output_shape),

    "lysis_rates":
        np.zeros(output_shape),

    "relative_abundance":
        np.zeros(output_shape)
}


# ============================================================
# 8. Run simulations
# ============================================================

for scenario_idx, scenario in enumerate(
    scenario_params
):

    sigma_base = scenario["sigma_base"]
    L_var = scenario["L_var"]
    max_inter = scenario["max_inter"]

    print(
        f"Scenario "
        f"{scenario_idx + 1}/{n_scenarios}"
        f" | sigma_base={sigma_base:.1f}"
        f" | L_var={L_var:.1f}"
        f" | max_inter={max_inter}"
    )

    for seed in range(1, n_seeds + 1):

        # Reproducible random values for this seed
        np.random.seed(seed)

        # ----------------------------------------------------
        # Species-specific lysis parameters
        # ----------------------------------------------------

        sigmas = (
            sigma_base
            + L_var
            * generate_valid_random(N)
        )

        lambdas = (
            lambda_base
            + L_var
            * generate_valid_random(N)
        )

        # This preserves the exact parameterization of the
        # previous no-debris model.
        #
        # If biologically required, the following two lines can
        # instead be activated to prevent negative values:
        #
        # sigmas = np.clip(sigmas, 0, None)
        # lambdas = np.clip(lambdas, 0, None)

        # ----------------------------------------------------
        # Interaction matrix
        # ----------------------------------------------------

        # max_Gs[-1] = 1.5
        current_max_inter = (
            max_Gs[-1] * max_inter
            if max_inter > 0
            else 0
        )

        params = param_generator_lv(
            N=N,
            connectedness=connectedness,
            neg_frac=neg_frac,
            max_neg=current_max_inter,
            max_pos=current_max_inter * 0.5
        )

        # Therefore:
        #
        # Negative interactions:
        #     U(-1.5 × max_inter, 0)
        #
        # Positive interactions:
        #     U(0, 0.75 × max_inter)

        # ----------------------------------------------------
        # Treatment
        # ----------------------------------------------------

        (
            t,
            y,
            net_rates,
            effective_rates,
            lysis_rates
        ) = simulate_treatment_timecourse(
            x0=x0,
            tend=tend,
            params=params,
            max_Gs=max_Gs,
            sigma=sigmas,
            lambda_val=lambdas,
            treatment_time=treatment_time
        )

        outputs[
            "population_data"
        ][:, :, seed - 1, scenario_idx] = y

        outputs[
            "growth_rates"
        ][:, :, seed - 1, scenario_idx] = (
            net_rates
        )

        outputs[
            "effective_growth_rates"
        ][:, :, seed - 1, scenario_idx] = (
            effective_rates
        )

        outputs[
            "lysis_rates"
        ][:, :, seed - 1, scenario_idx] = (
            lysis_rates
        )

        treatment_total = y.sum(
            axis=1,
            keepdims=True
        )

        outputs[
            "relative_abundance"
        ][:, :, seed - 1, scenario_idx] = (
            np.divide(
                y,
                treatment_total,
                out=np.zeros_like(y),
                where=treatment_total > 0
            )
        )

        # ----------------------------------------------------
        # Dilution control
        # ----------------------------------------------------

        (
            t_ctrl,
            y_ctrl,
            net_rates_ctrl,
            effective_rates_ctrl,
            lysis_rates_ctrl
        ) = simulate_control_timecourse(
            x0=x0,
            tend=tend,
            params=params,
            max_Gs=max_Gs,
            dilution_time=dilution_time,
            dilution_factor=dilution_factor
        )

        # Confirm identical output time grids
        if not np.allclose(t, t_ctrl):
            raise RuntimeError(
                "Treatment and control time grids differ."
            )

        control_outputs[
            "population_data"
        ][:, :, seed - 1, scenario_idx] = (
            y_ctrl
        )

        control_outputs[
            "growth_rates"
        ][:, :, seed - 1, scenario_idx] = (
            net_rates_ctrl
        )

        control_outputs[
            "effective_growth_rates"
        ][:, :, seed - 1, scenario_idx] = (
            effective_rates_ctrl
        )

        control_outputs[
            "lysis_rates"
        ][:, :, seed - 1, scenario_idx] = (
            lysis_rates_ctrl
        )

        control_total = y_ctrl.sum(
            axis=1,
            keepdims=True
        )

        control_outputs[
            "relative_abundance"
        ][:, :, seed - 1, scenario_idx] = (
            np.divide(
                y_ctrl,
                control_total,
                out=np.zeros_like(y_ctrl),
                where=control_total > 0
            )
        )


# ============================================================
# 9. Convert scenario metadata to simple arrays
# ============================================================

scenario_sigma_base = np.array(
    [
        scenario["sigma_base"]
        for scenario in scenario_params
    ],
    dtype=float
)

scenario_L_var = np.array(
    [
        scenario["L_var"]
        for scenario in scenario_params
    ],
    dtype=float
)

scenario_max_inter = np.array(
    [
        scenario["max_inter"]
        for scenario in scenario_params
    ],
    dtype=int
)


# ============================================================
# 10. Save all results
# ============================================================

output_filename = (
    "simulation_results_sigma_scan_10seeds_dilution25.npz"
)

np.savez_compressed(
    output_filename,

    # General metadata
    time_points=t,
    seed_values=np.arange(1, n_seeds + 1),
    species_index=np.arange(1, N + 1),

    # Scenario metadata
    sigma_base_values=sigma_base_values,
    L_var_values=np.array(L_var_values),
    max_inter_values=np.array(max_inter_values),

    scenario_sigma_base=scenario_sigma_base,
    scenario_L_var=scenario_L_var,
    scenario_max_inter=scenario_max_inter,

    # Fixed parameters
    initial_abundance=x0,
    intrinsic_growth_rates=max_Gs,
    lambda_base=lambda_base,
    connectedness=connectedness,
    negative_interaction_fraction=neg_frac,
    treatment_time=treatment_time,
    dilution_time=dilution_time,
    dilution_factor=dilution_factor,

    # Treatment outputs
    population_data=
        outputs["population_data"],

    growth_rates=
        outputs["growth_rates"],

    effective_growth_rates=
        outputs["effective_growth_rates"],

    lysis_rates=
        outputs["lysis_rates"],

    relative_abundance=
        outputs["relative_abundance"],

    # Control outputs
    control_population_data=
        control_outputs["population_data"],

    control_growth_rates=
        control_outputs["growth_rates"],

    control_effective_growth_rates=
        control_outputs[
            "effective_growth_rates"
        ],

    control_lysis_rates=
        control_outputs["lysis_rates"],

    control_relative_abundance=
        control_outputs[
            "relative_abundance"
        ]
)

print("\nFinished.")
print(f"Results saved to: {output_filename}")
print(f"Treatment array shape: {outputs['population_data'].shape}")
print(
    f"Control array shape: "
    f"{control_outputs['population_data'].shape}"
)

Number of sigma_base values: 11
Number of L_var values: 3
Number of max_inter values: 3
Total scenarios: 99
Scenario-seed combinations: 990
Treatment + control timecourses: 1980
Scenario 1/99 | sigma_base=0.5 | L_var=0.0 | max_inter=1
Scenario 2/99 | sigma_base=0.5 | L_var=0.0 | max_inter=5
Scenario 3/99 | sigma_base=0.5 | L_var=0.0 | max_inter=10
Scenario 4/99 | sigma_base=0.5 | L_var=0.1 | max_inter=1
Scenario 5/99 | sigma_base=0.5 | L_var=0.1 | max_inter=5
Scenario 6/99 | sigma_base=0.5 | L_var=0.1 | max_inter=10
Scenario 7/99 | sigma_base=0.5 | L_var=0.2 | max_inter=1
Scenario 8/99 | sigma_base=0.5 | L_var=0.2 | max_inter=5
Scenario 9/99 | sigma_base=0.5 | L_var=0.2 | max_inter=10
Scenario 10/99 | sigma_base=0.6 | L_var=0.0 | max_inter=1
Scenario 11/99 | sigma_base=0.6 | L_var=0.0 | max_inter=5
Scenario 12/99 | sigma_base=0.6 | L_var=0.0 | max_inter=10
Scenario 13/99 | sigma_base=0.6 | L_var=0.1 | max_inter=1
Scenario 14/99 | sigma_base=0.6 | L_var=0.1 | max_inter=5
Scenario 15/99 

In [4]:

import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import linregress
from sklearn.metrics import r2_score

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize


# =====================================================
# Shannon diversity
# =====================================================
def calculate_shannon_from_population(
        population_data,
        absolute_extinction_threshold=1e-6
):
    """
    Calculate Shannon diversity from absolute population data.

    Once a species falls to or below the extinction threshold,
    it is treated as extinct for all subsequent time points.
    """

    shannon_data = np.full(
        (
            len(plot_indices),
            n_seeds,
            n_scenarios
        ),
        np.nan
    )

    for scenario_idx in range(n_scenarios):

        for seed_idx in range(n_seeds):

            # Full selected trajectory:
            # time × species
            population_trajectory = (
                population_data[
                    plot_indices,
                    :,
                    seed_idx,
                    scenario_idx
                ].copy()
            )

            # Track whether each species has already gone extinct
            extinct = np.zeros(
                n_species,
                dtype=bool
            )

            for output_t_idx in range(
                    len(plot_indices)
            ):

                current_population = (
                    population_trajectory[
                        output_t_idx
                    ].copy()
                )

                # Species crossing the absolute threshold
                # are considered extinct
                newly_extinct = (
                    ~np.isfinite(
                        current_population
                    )
                    |
                    (
                        current_population
                        <= absolute_extinction_threshold
                    )
                )

                # Extinction is irreversible
                extinct = (
                    extinct
                    |
                    newly_extinct
                )

                current_population[
                    extinct
                ] = 0.0

                total_population = np.sum(
                    current_population
                )

                if (
                    not np.isfinite(
                        total_population
                    )
                    or total_population <= 0
                ):
                    continue

                relative_abundance = (
                    current_population
                    / total_population
                )

                positive = (
                    relative_abundance > 0
                )

                shannon_data[
                    output_t_idx,
                    seed_idx,
                    scenario_idx
                ] = -np.sum(
                    relative_abundance[
                        positive
                    ]
                    *
                    np.log(
                        relative_abundance[
                            positive
                        ]
                    )
                )

    return shannon_data


# =====================================================
# Safe linear regression
# =====================================================

def calculate_slope_and_r2(x, y):
    """
    Fit the across-species relationship at one time point:

        lysis rate = slope * effective growth rate + intercept

    Returns
    -------
    slope : float
    r2 : float
    """

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if x.size < 2:
        return np.nan, np.nan

    if np.allclose(x, x[0]):
        return np.nan, np.nan

    result = linregress(
        x,
        y
    )

    y_pred = (
        result.slope * x
        + result.intercept
    )

    r2 = r2_score(
        y,
        y_pred
    )

    return result.slope, r2


# =====================================================
# Load simulation results
# =====================================================

input_file = (
    "simulation_results_sigma_scan_"
    "10seeds_dilution25.npz"
)

data = np.load(
    input_file,
    allow_pickle=True
)

print("Variables saved in the NPZ file:")

for key in data.files:
    print(key)


# Treatment rates used for fitting
treatment_effective_growth_rates = data[
    "effective_growth_rates"
]

treatment_lysis_rates = data[
    "lysis_rates"
]


# Absolute population trajectories
treatment_population_data = data[
    "population_data"
]

control_population_data = data[
    "control_population_data"
]


# Relative abundance trajectories
treatment_relative_abundance = data[
    "relative_abundance"
]

control_relative_abundance = data[
    "control_relative_abundance"
]


time_points = data[
    "time_points"
]


# Scenario metadata
scenario_sigma_base = data[
    "scenario_sigma_base"
]

scenario_L_var = data[
    "scenario_L_var"
]

scenario_max_inter = data[
    "scenario_max_inter"
]


(
    n_time,
    n_species,
    n_seeds,
    n_scenarios
) = treatment_effective_growth_rates.shape


print("\nSimulation dimensions:")
print(f"Time points: {n_time}")
print(f"Species: {n_species}")
print(f"Seeds: {n_seeds}")
print(f"Scenarios: {n_scenarios}")


# =====================================================
# Dimension checks
# =====================================================

if not (
    len(scenario_sigma_base)
    == len(scenario_L_var)
    == len(scenario_max_inter)
    == n_scenarios
):
    raise ValueError(
        "Scenario metadata lengths do not match "
        "the number of scenarios."
    )


expected_shape = (
    n_time,
    n_species,
    n_seeds,
    n_scenarios
)

arrays_to_check = {
    "treatment population":
        treatment_population_data,

    "control population":
        control_population_data,

    "treatment relative abundance":
        treatment_relative_abundance,

    "control relative abundance":
        control_relative_abundance,

    "treatment lysis rates":
        treatment_lysis_rates
}

for array_name, array_value in arrays_to_check.items():

    if array_value.shape != expected_shape:

        raise ValueError(
            f"{array_name} has unexpected shape "
            f"{array_value.shape}; expected {expected_shape}."
        )


# =====================================================
# Parameter levels
# =====================================================

sigma_levels = np.sort(
    np.unique(
        np.round(
            scenario_sigma_base,
            8
        )
    )
)

L_var_levels = np.sort(
    np.unique(
        np.round(
            scenario_L_var,
            8
        )
    )
)

max_inter_levels = np.sort(
    np.unique(
        scenario_max_inter
    )
)


print("\nSigma levels:")
print(sigma_levels)

print("\nL_var levels:")
print(L_var_levels)

print("\nmax_inter levels:")
print(max_inter_levels)


# =====================================================
# Time settings
# =====================================================

fit_time = 6.0

fit_idx = np.argmin(
    np.abs(
        time_points - fit_time
    )
)

actual_fit_time = time_points[
    fit_idx
]

print(
    f"\nTreatment regression evaluated at "
    f"{actual_fit_time:.2f} h"
)


plot_start_time = 4.0
plot_end_time = 8.0

time_mask = (
    (time_points >= plot_start_time)
    &
    (time_points <= plot_end_time)
)

plot_indices = np.where(
    time_mask
)[0]


# =====================================================
# Find scenario index
# =====================================================

def find_scenario_index(
        sigma_base,
        L_var,
        max_inter
):
    """
    Return the unique scenario index for:

        sigma_base × L_var × max_inter
    """

    matches = np.where(
        np.isclose(
            scenario_sigma_base,
            sigma_base
        )
        &
        np.isclose(
            scenario_L_var,
            L_var
        )
        &
        np.isclose(
            scenario_max_inter,
            max_inter
        )
    )[0]

    if len(matches) == 0:
        raise ValueError(
            "No scenario found for "
            f"sigma_base={sigma_base}, "
            f"L_var={L_var}, "
            f"max_inter={max_inter}."
        )

    if len(matches) > 1:
        raise ValueError(
            "Multiple scenarios found for "
            f"sigma_base={sigma_base}, "
            f"L_var={L_var}, "
            f"max_inter={max_inter}."
        )

    return int(
        matches[0]
    )


# =====================================================
# Treatment OD filtering at 8 h
# =====================================================

minimum_total_od = 0.01
od_check_time = 8.0

od_check_idx = np.argmin(
    np.abs(time_points - od_check_time)
)

actual_od_check_time = time_points[
    od_check_idx
]

# Sum absolute population across species:
# time × species × seed × scenario -> time × seed × scenario

treatment_total_od = np.sum(
    treatment_population_data,
    axis=1
)

# A treatment simulation passes the OD filter only when its
# total OD is finite and strictly greater than 0.01 at 8 h.
valid_treatment_od_at_8h = (
    np.isfinite(
        treatment_total_od[
            od_check_idx,
            :,
            :
        ]
    )
    &
    (
        treatment_total_od[
            od_check_idx,
            :,
            :
        ] > minimum_total_od
    )
)

print(
    f"\nTreatment simulations with total OD > "
    f"{minimum_total_od:g} at "
    f"{actual_od_check_time:g} h: "
    f"{np.sum(valid_treatment_od_at_8h)}/"
    f"{valid_treatment_od_at_8h.size}"
)


# =====================================================
# Fit treatment slopes and R² at 6 h
# =====================================================

treatment_slopes = np.full(
    (
        n_seeds,
        n_scenarios
    ),
    np.nan
)

treatment_r2_values = np.full(
    (
        n_seeds,
        n_scenarios
    ),
    np.nan
)


for scenario_idx in range(
        n_scenarios
):

    for seed_idx in range(
            n_seeds
    ):

        x_treatment = (
            treatment_effective_growth_rates[
                fit_idx,
                :,
                seed_idx,
                scenario_idx
            ]
        )

        y_treatment = (
            treatment_lysis_rates[
                fit_idx,
                :,
                seed_idx,
                scenario_idx
            ]
        )

        slope, r2 = calculate_slope_and_r2(
            x_treatment,
            y_treatment
        )

        treatment_slopes[
            seed_idx,
            scenario_idx
        ] = slope

        treatment_r2_values[
            seed_idx,
            scenario_idx
        ] = r2


# =====================================================
# Sigma-level matched filtering
# =====================================================

minimum_slope = 0.5
maximum_slope = 1.5
minimum_r2 = 0.6


# Dimensions:
# seed × scenario
#
# A treatment sigma × seed simulation passes only when:
#   1. total treatment OD is > 0.01 at 8 h
#   2. the treatment G/L slope fitted at 6 h is finite
#   3. 0.5 <= treatment slope <= 1.5
#   4. treatment R² > 0.6
#
# The paired control is selected only by the corresponding
# treatment simulation; control OD and control fitting are not
# used as additional filters.

valid_matched_simulation = (
    valid_treatment_od_at_8h
    &
    np.isfinite(
        treatment_slopes
    )
    &
    np.isfinite(
        treatment_r2_values
    )
    &
    (
        treatment_slopes
        >= minimum_slope
    )
    &
    (
        treatment_slopes
        <= maximum_slope
    )
    &
    (
        treatment_r2_values
        > minimum_r2
    )
)


all_valid_treatment_slopes = (
    treatment_slopes[
        valid_matched_simulation
    ]
)


if all_valid_treatment_slopes.size == 0:

    raise ValueError(
        "No matched treatment/control simulations passed "
        "all filters: "
        f"{minimum_slope} <= slope <= {maximum_slope}, "
        f"R² > {minimum_r2}, and "
        f"treatment OD > {minimum_total_od} at "
        f"{actual_od_check_time:g} h."
    )


print(
    "\nSigma × seed simulations passing all matched filters: "
    f"{np.sum(valid_matched_simulation)}/"
    f"{valid_matched_simulation.size}"
)

print(
    "Valid treatment slope range: "
    f"{all_valid_treatment_slopes.min():.4f}–"
    f"{all_valid_treatment_slopes.max():.4f}"
)


# =====================================================
# Shared color scale
# =====================================================

norm = Normalize(
    vmin=minimum_slope,
    vmax=maximum_slope
)

cmap = plt.cm.viridis


# =====================================================
# Calculate Shannon trajectories
# =====================================================

def calculate_all_shannon_trajectories(
        abundance_data
):
    """
    Return Shannon trajectories with dimensions:

        selected time × seed × scenario
    """

    shannon_data = np.full(
        (
            len(plot_indices),
            n_seeds,
            n_scenarios
        ),
        np.nan
    )

    for scenario_idx in range(
            n_scenarios
    ):

        for seed_idx in range(
                n_seeds
        ):

            for output_t_idx, source_t_idx in enumerate(
                    plot_indices
            ):

                p = abundance_data[
                    source_t_idx,
                    :,
                    seed_idx,
                    scenario_idx
                ]

                shannon_data[
                    output_t_idx,
                    seed_idx,
                    scenario_idx
                ] = shannon_index(p)

    return shannon_data


print(
    "\nCalculating treatment Shannon trajectories..."
)


treatment_shannon = (
    calculate_shannon_from_population(
        treatment_population_data,
        absolute_extinction_threshold=1e-6
    )
)

control_shannon = (
    calculate_shannon_from_population(
        control_population_data,
        absolute_extinction_threshold=1e-6
    )
)

# =====================================================
# Panel-level matched seed selection
# =====================================================

def get_panel_valid_seeds(
        L_var,
        max_inter
):
    """
    For one L_var × max_inter panel, identify seeds with
    at least one sigma_base that passes every matched filter.

    Returns
    -------
    panel_valid_seed_mask
        Boolean array of length n_seeds.

    valid_scenarios_by_seed
        Dictionary:
            seed_idx -> list of qualified scenario indices
    """

    panel_valid_seed_mask = np.zeros(
        n_seeds,
        dtype=bool
    )

    valid_scenarios_by_seed = {
        seed_idx: []
        for seed_idx in range(n_seeds)
    }


    for seed_idx in range(
            n_seeds
    ):

        for sigma_base in sigma_levels:

            scenario_idx = find_scenario_index(
                sigma_base=sigma_base,
                L_var=L_var,
                max_inter=max_inter
            )

            if valid_matched_simulation[
                seed_idx,
                scenario_idx
            ]:

                panel_valid_seed_mask[
                    seed_idx
                ] = True

                valid_scenarios_by_seed[
                    seed_idx
                ].append(
                    scenario_idx
                )


    return (
        panel_valid_seed_mask,
        valid_scenarios_by_seed
    )


# =====================================================
# Shared page formatting
# =====================================================

def format_page(
        fig,
        axes,
        condition_name
):

    for col_idx, max_inter in enumerate(
            max_inter_levels
    ):

        axes[
            0,
            col_idx
        ].set_title(
            f"max_inter = {max_inter:g}",
            fontsize=16
        )


    for row_idx, L_var in enumerate(
            L_var_levels
    ):

        axes[
            row_idx,
            0
        ].set_ylabel(
            f"L_var = {L_var:g}\n"
            "Shannon diversity",
            fontsize=14
        )


    for ax in axes[-1, :]:

        ax.set_xlabel(
            "Time (h)",
            fontsize=14
        )


    for ax in axes.ravel():

        ax.set_xlim(
            plot_start_time,
            plot_end_time
        )

        ax.set_ylim(
            0,
            4.2
        )

        ax.tick_params(
            axis="both",
            labelsize=11
        )


    fig.subplots_adjust(
        right=0.86,
        top=0.90,
        left=0.11,
        bottom=0.09,
        wspace=0.15,
        hspace=0.17
    )


    cax = fig.add_axes(
        [
            0.89,
            0.16,
            0.025,
            0.68
        ]
    )


    scalar_map = ScalarMappable(
        norm=norm,
        cmap=cmap
    )

    scalar_map.set_array([])


    colorbar = fig.colorbar(
        scalar_map,
        cax=cax
    )

    colorbar.set_label(
        (
            "Treatment growth–lysis slope "
            f"at {actual_fit_time:g} h"
        ),
        fontsize=14
    )

    colorbar.ax.tick_params(
        labelsize=11
    )


    fig.suptitle(
        condition_name,
        fontsize=20,
        y=0.97
    )


# =====================================================
# Treatment page
# =====================================================

def make_treatment_page():
    """
    Treatment page.

    Within each panel:
      - The seed set is identical to the control seed set.
      - A seed is included only if at least one sigma passes
        all matched treatment/control filters.
      - For an included seed, only qualified sigma trajectories
        are plotted.
      - Each qualified sigma × seed is one line.
    """

    n_rows = len(
        L_var_levels
    )

    n_cols = len(
        max_inter_levels
    )


    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(7, 6),
        sharex=True,
        sharey=True,
        squeeze=False
    )


    for row_idx, L_var in enumerate(
            L_var_levels
    ):

        for col_idx, max_inter in enumerate(
                max_inter_levels
        ):

            ax = axes[
                row_idx,
                col_idx
            ]


            (
                panel_valid_seed_mask,
                valid_scenarios_by_seed
            ) = get_panel_valid_seeds(
                L_var=L_var,
                max_inter=max_inter
            )


            valid_seed_indices = np.where(
                panel_valid_seed_mask
            )[0]


            curve_count = 0


            for seed_idx in valid_seed_indices:

                for scenario_idx in (
                        valid_scenarios_by_seed[
                            seed_idx
                        ]
                ):

                    treatment_slope = (
                        treatment_slopes[
                            seed_idx,
                            scenario_idx
                        ]
                    )

                    shannon_values = (
                        treatment_shannon[
                            :,
                            seed_idx,
                            scenario_idx
                        ]
                    )

                    if not np.any(
                        np.isfinite(
                            shannon_values
                        )
                    ):
                        continue


                    ax.plot(
                        time_points[
                            plot_indices
                        ],
                        shannon_values,
                        color=cmap(
                            norm(
                                treatment_slope
                            )
                        ),
                        alpha=0.35,
                        linewidth=1.1
                    )

                    curve_count += 1

            ax.grid(
                alpha=0.20,
                linewidth=0.7
            )


    format_page(
        fig=fig,
        axes=axes,
        condition_name="Treatment"
    )

    return fig


# =====================================================
# Control page
# =====================================================

def make_control_page():
    """
    Control page.

    Within each panel:
      - Uses exactly the treatment-qualified seed × sigma pairs.
      - Each control curve is paired one-to-one with a qualified
        treatment curve.
      - The control line uses the same color as its corresponding
        treatment line, based on that treatment slope at 6 h.
      - Control trajectories can overlap because control does not
        depend on sigma_base, but the one-to-one plotting preserves
        exact treatment/control color matching.
    """

    n_rows = len(
        L_var_levels
    )

    n_cols = len(
        max_inter_levels
    )


    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(7, 6),
        sharex=True,
        sharey=True,
        squeeze=False
    )


    for row_idx, L_var in enumerate(
            L_var_levels
    ):

        for col_idx, max_inter in enumerate(
                max_inter_levels
        ):

            ax = axes[
                row_idx,
                col_idx
            ]


            (
                panel_valid_seed_mask,
                valid_scenarios_by_seed
            ) = get_panel_valid_seeds(
                L_var=L_var,
                max_inter=max_inter
            )


            valid_seed_indices = np.where(
                panel_valid_seed_mask
            )[0]


            curve_count = 0


            for seed_idx in valid_seed_indices:

                qualified_scenario_indices = (
                    valid_scenarios_by_seed[
                        seed_idx
                    ]
                )

                for scenario_idx in qualified_scenario_indices:

                    # Use the treatment slope from this exact
                    # seed × sigma scenario for color assignment.
                    treatment_slope = (
                        treatment_slopes[
                            seed_idx,
                            scenario_idx
                        ]
                    )

                    shannon_values = (
                        control_shannon[
                            :,
                            seed_idx,
                            scenario_idx
                        ]
                    )

                    if not np.any(
                        np.isfinite(
                            shannon_values
                        )
                    ):
                        continue

                    ax.plot(
                        time_points[
                            plot_indices
                        ],
                        shannon_values,
                        color=cmap(
                            norm(
                                treatment_slope
                            )
                        ),
                        alpha=0.70,
                        linewidth=1.5
                    )

                    curve_count += 1


            ax.grid(
                alpha=0.20,
                linewidth=0.7
            )


    format_page(
        fig=fig,
        axes=axes,
        condition_name="Dilution control"
    )

    return fig


# =====================================================
# Verify panel-level seed matching
# =====================================================

print(
    "\nPanel-level matched seed summary:"
)

for L_var in L_var_levels:

    for max_inter in max_inter_levels:

        panel_valid_seed_mask, _ = (
            get_panel_valid_seeds(
                L_var=L_var,
                max_inter=max_inter
            )
        )

        valid_seed_numbers = (
            np.where(
                panel_valid_seed_mask
            )[0]
            + 1
        )

        print(
            f"L_var={L_var:g}, "
            f"max_inter={max_inter:g}: "
            f"seeds={valid_seed_numbers.tolist()}"
        )


# =====================================================
# Save two-page PDF
# =====================================================

pdf_file = (
    "Shannon_matched_panel_seeds_"
    "treatment_all_sigma_"
    "control_once_per_seed_"
    "mean_treatment_slope_"
    "OD8hgt0.01_R2gt0.6_slope0.5to1.5.pdf"
)


with PdfPages(
        pdf_file
) as pdf:

    fig_treatment = make_treatment_page()

    pdf.savefig(
        fig_treatment,
        bbox_inches="tight"
    )

    plt.close(
        fig_treatment
    )


    fig_control = make_control_page()

    pdf.savefig(
        fig_control,
        bbox_inches="tight"
    )

    plt.close(
        fig_control
    )


print(
    f"\nSaved to: {pdf_file}"
)


Variables saved in the NPZ file:
time_points
seed_values
species_index
sigma_base_values
L_var_values
max_inter_values
scenario_sigma_base
scenario_L_var
scenario_max_inter
initial_abundance
intrinsic_growth_rates
lambda_base
connectedness
negative_interaction_fraction
treatment_time
dilution_time
dilution_factor
population_data
growth_rates
effective_growth_rates
lysis_rates
relative_abundance
control_population_data
control_growth_rates
control_effective_growth_rates
control_lysis_rates
control_relative_abundance

Simulation dimensions:
Time points: 111
Species: 57
Seeds: 10
Scenarios: 99

Sigma levels:
[0.5 0.6 0.7 0.8 0.9 1.  1.1 1.2 1.3 1.4 1.5]

L_var levels:
[0.  0.1 0.2]

max_inter levels:
[ 1  5 10]

Treatment regression evaluated at 6.00 h

Treatment simulations with total OD > 0.01 at 8 h: 990/990

Sigma × seed simulations passing all matched filters: 564/990
Valid treatment slope range: 0.5018–1.5000

Calculating treatment Shannon trajectories...

Panel-level matched seed s

In [5]:
# ============================================================
# Shannon diversity at 8 h
#
# Two statistical analyses:
#
# 1. Group by:
#       L_var + max_inter
#    Pool all sigma_base values and seeds within each group
#
# 2. Group by:
#       sigma_base + L_var + max_inter
#
# For each group:
#    paired Treatment vs Untreated comparison
#    using Wilcoxon signed-rank test
#
# Also report:
#    which group has higher mean Shannon diversity
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon, ttest_rel
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. Load simulation results
# ============================================================

result_file = "simulation_results_sigma_scan_10seeds_dilution25.npz"

data = np.load(
    result_file,
    allow_pickle=True
)

time_points = data["time_points"]

treat_ra = data["relative_abundance"]
control_ra = data["control_relative_abundance"]

scenario_sigma_base = data["scenario_sigma_base"]
scenario_L_var = data["scenario_L_var"]
scenario_max_inter = data["scenario_max_inter"]


# ============================================================
# 2. Find exact 8 h time point
# ============================================================

time_target = 8.0

time_idx = np.argmin(
    np.abs(time_points - time_target)
)

if not np.isclose(
    time_points[time_idx],
    time_target
):
    raise ValueError(
        f"8 h was not found exactly. "
        f"Closest time is "
        f"{time_points[time_idx]} h."
    )

print(
    f"Using time point: "
    f"{time_points[time_idx]} h"
)


# ============================================================
# 3. Shannon diversity function
#
# Input:
#     species × seeds
#
# Output:
#     one Shannon value per seed
# ============================================================

def shannon_by_seed(relative_abundance):

    relative_abundance = np.asarray(
        relative_abundance,
        dtype=float
    )

    # Numerical protection
    relative_abundance = np.maximum(
        relative_abundance,
        0
    )

    # Normalize to relative abundance
    total = relative_abundance.sum(
        axis=0,
        keepdims=True
    )

    p = np.divide(
        relative_abundance,
        total,
        out=np.zeros_like(
            relative_abundance
        ),
        where=total > 0
    )

    log_p = np.zeros_like(p)

    positive = p > 0

    log_p[positive] = np.log(
        p[positive]
    )

    shannon = -np.sum(
        p * log_p,
        axis=0
    )

    return shannon


# ============================================================
# 4. Build seed-level table
#
# Each row =
# one scenario × one seed
#
# Treatment and Untreated remain paired in the same row.
# ============================================================

seed_level_rows = []

n_scenarios = treat_ra.shape[3]

for scenario_idx in range(
    n_scenarios
):

    # --------------------------
    # Shannon at 8 h
    # --------------------------

    treat_shannon = shannon_by_seed(
        treat_ra[
            time_idx,
            :,
            :,
            scenario_idx
        ]
    )

    control_shannon = shannon_by_seed(
        control_ra[
            time_idx,
            :,
            :,
            scenario_idx
        ]
    )

    # --------------------------
    # Add one row per seed
    # --------------------------

    for seed_idx in range(
        len(treat_shannon)
    ):

        seed_level_rows.append(
            {
                "scenario_index":
                    scenario_idx + 1,

                "sigma_base":
                    scenario_sigma_base[
                        scenario_idx
                    ],

                "L_var":
                    scenario_L_var[
                        scenario_idx
                    ],

                "max_inter":
                    scenario_max_inter[
                        scenario_idx
                    ],

                "seed":
                    seed_idx + 1,

                "treatment_shannon":
                    treat_shannon[
                        seed_idx
                    ],

                "untreated_shannon":
                    control_shannon[
                        seed_idx
                    ],

                "difference":
                    treat_shannon[
                        seed_idx
                    ]
                    -
                    control_shannon[
                        seed_idx
                    ]
            }
        )


seed_level_df = pd.DataFrame(
    seed_level_rows
)


# Save raw seed-level values
seed_level_df.to_csv(
    "Shannon_8h_seed_level.csv",
    index=False
)


print(
    "\nSeed-level table:"
)

print(
    seed_level_df.head(
        20
    ).to_string(
        index=False
    )
)


# ============================================================
# 5. Function for paired statistics
# ============================================================

def calculate_paired_stats(
    df,
    group_cols
):

    """
    Perform paired Treatment vs Untreated
    statistics within groups defined by
    group_cols.

    Pairing is preserved because treatment
    and untreated values from the same
    scenario/seed are stored in the same row.
    """

    summary_rows = []

    grouped = df.groupby(
        group_cols,
        dropna=False,
        sort=True
    )

    for group_values, group_df in grouped:

        # ----------------------------------------
        # Ensure group_values is always a tuple
        # ----------------------------------------

        if not isinstance(
            group_values,
            tuple
        ):
            group_values = (
                group_values,
            )

        # ----------------------------------------
        # Extract paired Shannon values
        # ----------------------------------------

        treatment = (
            group_df[
                "treatment_shannon"
            ]
            .to_numpy(
                dtype=float
            )
        )

        untreated = (
            group_df[
                "untreated_shannon"
            ]
            .to_numpy(
                dtype=float
            )
        )

        # ----------------------------------------
        # Remove non-finite pairs
        # ----------------------------------------

        valid = (
            np.isfinite(treatment)
            &
            np.isfinite(untreated)
        )

        treatment = treatment[
            valid
        ]

        untreated = untreated[
            valid
        ]

        differences = (
            treatment
            -
            untreated
        )

        n_pairs = len(
            treatment
        )

        # ----------------------------------------
        # Descriptive statistics
        # ----------------------------------------

        if n_pairs > 0:

            treatment_mean = np.mean(
                treatment
            )

            untreated_mean = np.mean(
                untreated
            )

            mean_difference = np.mean(
                differences
            )

        else:

            treatment_mean = np.nan
            untreated_mean = np.nan
            mean_difference = np.nan


        if n_pairs >= 2:

            treatment_sd = np.std(
                treatment,
                ddof=1
            )

            untreated_sd = np.std(
                untreated,
                ddof=1
            )

        else:

            treatment_sd = np.nan
            untreated_sd = np.nan


        # ----------------------------------------
        # Which group is higher?
        # ----------------------------------------

        if (
            np.isnan(
                treatment_mean
            )
            or
            np.isnan(
                untreated_mean
            )
        ):

            higher_group = "NA"

        elif np.isclose(
            treatment_mean,
            untreated_mean
        ):

            higher_group = "Equal"

        elif (
            treatment_mean
            >
            untreated_mean
        ):

            higher_group = "Treatment"

        else:

            higher_group = "Untreated"


        # ----------------------------------------
        # Paired Wilcoxon signed-rank test
        # ----------------------------------------

        if (
            n_pairs >= 2
            and
            not np.allclose(
                differences,
                0
            )
        ):

            wilcoxon_result = wilcoxon(
                treatment,
                untreated,
                alternative="two-sided",
                zero_method="wilcox"
            )

            wilcoxon_stat = (
                wilcoxon_result.statistic
            )

            wilcoxon_p = (
                wilcoxon_result.pvalue
            )

        elif (
            n_pairs >= 2
            and
            np.allclose(
                differences,
                0
            )
        ):

            wilcoxon_stat = 0.0
            wilcoxon_p = 1.0

        else:

            wilcoxon_stat = np.nan
            wilcoxon_p = np.nan


        # ----------------------------------------
        # Paired t-test
        # Reference only
        # ----------------------------------------

        if (
            n_pairs >= 2
            and
            not np.allclose(
                differences,
                0
            )
        ):

            ttest_result = ttest_rel(
                treatment,
                untreated,
                nan_policy="omit"
            )

            paired_t_stat = (
                ttest_result.statistic
            )

            paired_t_p = (
                ttest_result.pvalue
            )

        elif (
            n_pairs >= 2
            and
            np.allclose(
                differences,
                0
            )
        ):

            paired_t_stat = 0.0
            paired_t_p = 1.0

        else:

            paired_t_stat = np.nan
            paired_t_p = np.nan


        # ----------------------------------------
        # Build output row
        # ----------------------------------------

        row = {}

        for col, value in zip(
            group_cols,
            group_values
        ):
            row[col] = value


        row.update(
            {
                "n_pairs":
                    n_pairs,

                "treatment_mean":
                    treatment_mean,

                "treatment_sd":
                    treatment_sd,

                "untreated_mean":
                    untreated_mean,

                "untreated_sd":
                    untreated_sd,

                "mean_treatment_minus_untreated":
                    mean_difference,

                "higher_group":
                    higher_group,

                "wilcoxon_statistic":
                    wilcoxon_stat,

                "wilcoxon_p":
                    wilcoxon_p,

                "paired_t_statistic":
                    paired_t_stat,

                "paired_t_p":
                    paired_t_p
            }
        )

        summary_rows.append(
            row
        )


    summary_df = pd.DataFrame(
        summary_rows
    )


    # ========================================================
    # FDR correction
    #
    # IMPORTANT:
    # FDR is performed separately for each analysis.
    # ========================================================

    summary_df[
        "wilcoxon_p_adj"
    ] = np.nan

    valid_p = (
        summary_df[
            "wilcoxon_p"
        ]
        .notna()
    )

    if valid_p.sum() > 0:

        summary_df.loc[
            valid_p,
            "wilcoxon_p_adj"
        ] = multipletests(
            summary_df.loc[
                valid_p,
                "wilcoxon_p"
            ],
            method="fdr_bh"
        )[1]


    # ========================================================
    # Significance symbols
    # ========================================================

    def significance_label(p):

        if pd.isna(p):
            return "NA"

        if p < 0.0001:
            return "****"

        elif p < 0.001:
            return "***"

        elif p < 0.01:
            return "**"

        elif p < 0.05:
            return "*"

        else:
            return "ns"


    summary_df[
        "significance"
    ] = (
        summary_df[
            "wilcoxon_p_adj"
        ]
        .apply(
            significance_label
        )
    )


    summary_df[
        "significant"
    ] = np.where(
        summary_df[
            "wilcoxon_p_adj"
        ] < 0.05,
        "Yes",
        "No"
    )


    return summary_df


# ============================================================
# 6. Analysis 1
#
# Group by:
#     L_var + max_inter
#
# sigma_base values are pooled together.
#
# Example:
#
# L_var = 0.2
# max_inter = 0.1
#
# sigma_base = 0.1, 0.2, 0.3...
#
# all their seeds are pooled into the same
# paired analysis.
# ============================================================

summary_Lvar_inter = (
    calculate_paired_stats(
        seed_level_df,
        group_cols=[
            "L_var",
            "max_inter"
        ]
    )
)


# ============================================================
# 7. Analysis 2
#
# Group by:
#     sigma_base + L_var + max_inter
# ============================================================

summary_full_parameter = (
    calculate_paired_stats(
        seed_level_df,
        group_cols=[
            "sigma_base",
            "L_var",
            "max_inter"
        ]
    )
)


# ============================================================
# 8. Formatting function for output
# ============================================================

def format_summary(
    summary_df
):

    output = summary_df.copy()


    # ----------------------------------------
    # Mean ± SD
    # ----------------------------------------

    output[
        "Treatment"
    ] = (
        output[
            "treatment_mean"
        ].map(
            lambda x:
            f"{x:.4f}"
            if pd.notna(x)
            else "NA"
        )
        +
        " ± "
        +
        output[
            "treatment_sd"
        ].map(
            lambda x:
            f"{x:.4f}"
            if pd.notna(x)
            else "NA"
        )
    )


    output[
        "Untreated"
    ] = (
        output[
            "untreated_mean"
        ].map(
            lambda x:
            f"{x:.4f}"
            if pd.notna(x)
            else "NA"
        )
        +
        " ± "
        +
        output[
            "untreated_sd"
        ].map(
            lambda x:
            f"{x:.4f}"
            if pd.notna(x)
            else "NA"
        )
    )


    # ----------------------------------------
    # Round mean difference
    # ----------------------------------------

    output[
        "mean_treatment_minus_untreated"
    ] = (
        output[
            "mean_treatment_minus_untreated"
        ]
        .round(4)
    )


    # ----------------------------------------
    # Format p values
    # ----------------------------------------

    output[
        "wilcoxon_p"
    ] = (
        output[
            "wilcoxon_p"
        ].map(
            lambda x:
            f"{x:.3e}"
            if pd.notna(x)
            else "NA"
        )
    )


    output[
        "wilcoxon_p_adj"
    ] = (
        output[
            "wilcoxon_p_adj"
        ].map(
            lambda x:
            f"{x:.3e}"
            if pd.notna(x)
            else "NA"
        )
    )


    return output


# ============================================================
# 9. Format both tables
# ============================================================

final_Lvar_inter = format_summary(
    summary_Lvar_inter
)

final_full_parameter = format_summary(
    summary_full_parameter
)


# ============================================================
# 10. Select columns
# ============================================================

final_Lvar_inter = (
    final_Lvar_inter[
        [
            "L_var",
            "max_inter",
            "n_pairs",
            "Treatment",
            "Untreated",
            "mean_treatment_minus_untreated",
            "higher_group",
            "wilcoxon_p",
            "wilcoxon_p_adj",
            "significance",
            "significant"
        ]
    ]
)


final_full_parameter = (
    final_full_parameter[
        [
            "sigma_base",
            "L_var",
            "max_inter",
            "n_pairs",
            "Treatment",
            "Untreated",
            "mean_treatment_minus_untreated",
            "higher_group",
            "wilcoxon_p",
            "wilcoxon_p_adj",
            "significance",
            "significant"
        ]
    ]
)


# ============================================================
# 11. Print results
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "ANALYSIS 1:"
)

print(
    "Grouped by L_var + max_inter"
)

print(
    "=" * 80
)

print(
    final_Lvar_inter.to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "ANALYSIS 2:"
)

print(
    "Grouped by sigma_base + "
    "L_var + max_inter"
)

print(
    "=" * 80
)

print(
    final_full_parameter.to_string(
        index=False
    )
)


# ============================================================
# 12. Save CSV files
# ============================================================

final_Lvar_inter.to_csv(
    "Shannon_8h_group_by_Lvar_maxinter.csv",
    index=False
)


final_full_parameter.to_csv(
    "Shannon_8h_group_by_sigma_Lvar_maxinter.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    "  Shannon_8h_group_by_Lvar_maxinter.csv"
)

print(
    "  Shannon_8h_group_by_sigma_Lvar_maxinter.csv"
)

print(
    "  Shannon_8h_seed_level.csv"
)

Using time point: 8.0 h

Seed-level table:
 scenario_index  sigma_base  L_var  max_inter  seed  treatment_shannon  untreated_shannon  difference
              1         0.5    0.0          1     1           3.413024           3.173605    0.239419
              1         0.5    0.0          1     2           3.454733           3.216962    0.237770
              1         0.5    0.0          1     3           3.404024           3.168200    0.235824
              1         0.5    0.0          1     4           3.411009           3.165007    0.246002
              1         0.5    0.0          1     5           3.411394           3.184791    0.226603
              1         0.5    0.0          1     6           3.399728           3.156024    0.243704
              1         0.5    0.0          1     7           3.375996           3.134359    0.241637
              1         0.5    0.0          1     8           3.419657           3.185216    0.234441
              1         0.5    0.0     